# fastText で未知語にもベクトルを与える

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/06_word_vectors/fasttext_train.ipynb)

NLP クックブック 06-02 ／ 単語ベクトル

word2vec は語彙にない単語を扱えない。文字 n-gram を使う fastText なら、未知語や表記ゆれにもベクトルを与えられる。

In [1]:
!pip install -q gensim

In [2]:
import nltk
nltk.download("brown", quiet=True)
from nltk.corpus import brown
from gensim.models import FastText, Word2Vec

sentences = [[w.lower() for w in s] for s in brown.sents()]
ft = FastText(sentences, vector_size=100, window=5, min_count=5, min_n=3, max_n=6, epochs=10, workers=2, seed=0)
w2v = Word2Vec(sentences, vector_size=100, window=5, min_count=5, epochs=10, workers=2, seed=0)

In [3]:
for w in ["governmental", "governmentally", "goverment", "unhappily"]:
    in_vocab = w in ft.wv.key_to_index
    print(f"{w:16} 語彙内: {in_vocab!s:6} fastText: {[x for x, _ in ft.wv.most_similar(w, topn=4)]}")
    if w not in w2v.wv:
        print(f"{'':16} word2vec: KeyError (語彙にない)")

governmental     語彙内: True   fastText: ['government', "government's", 'congregational', 'international']
governmentally   語彙内: False  fastText: ['government', 'nationally', 'governmental', 'pro-communist']
                 word2vec: KeyError (語彙にない)
goverment        語彙内: False  fastText: ['government', 'commitment', 'punishment', 'abandonment']
                 word2vec: KeyError (語彙にない)
unhappily        語彙内: True   fastText: ['visibly', 'terribly', 'hopelessly', 'truly']


- 単語ベクトル = 単語自身 + 文字 n-gram（`min_n`〜`max_n` 文字）のベクトルの和。綴りが似ている語は近くなる。
- 形態が豊かな言語（ドイツ語、トルコ語、フィンランド語など）やタイポの多いテキストで特に有効。
- Meta 公開の学習済みモデル（157 言語、`cc.ja.300.bin` など）は `fasttext` パッケージで読める（各 4GB 以上）。